<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M05/M05_Lab2_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">📚 M05 Lab 2 — Building a RAG Pipeline</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐⭐ Difficulty: Advanced &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Build a complete <b>RAG pipeline</b>: load → split → embed → store → retrieve → generate</li>
    <li>Use <b>RecursiveCharacterTextSplitter</b> to chunk documents</li>
    <li>Create a <b>retrieval chain</b> with LangChain</li>
    <li>Compare responses <b>WITH RAG</b> vs <b>WITHOUT RAG</b> side by side</li>
    <li>Evaluate retrieval quality and understand when RAG fails</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai langchain langchain-openai langchain-community chromadb
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
import os
from google.colab import userdata
from dads5250 import setup_openai, show_info, quiz

client = setup_openai()
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ The RAG Architecture</h2>
</div>

**RAG (Retrieval-Augmented Generation)** solves a critical LLM limitation: **hallucination**. Instead of relying solely on training data, RAG retrieves relevant documents and includes them as context.

```
📄 Document  →  ✂️ Split into chunks  →  🧮 Embed  →  🗄️ Store in vector DB
                                                                    ↓
❓ User Query  →  🧮 Embed query  →  🔍 Find similar chunks  →  📝 Generate grounded answer
```

| Without RAG | With RAG |
|------------|----------|
| LLM guesses from training data | LLM answers from YOUR documents |
| May hallucinate facts | Grounded in real data |
| Can't access private data | Works with any document |
| Knowledge cutoff date | Always up to date |

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Load & Chunk the Document</h2>
</div>

We'll use a **sample company report** as our knowledge base. In production, this could be PDFs, web pages, databases, or any text source.

In [ ]:
# ============================================================
# 📄 Sample Document: TechCorp Annual Report 2024
# ============================================================

company_report = """
TECHCORP INC. — ANNUAL REPORT 2024

EXECUTIVE SUMMARY
TechCorp Inc. reported record revenue of $4.2 billion in fiscal year 2024, representing a 23% increase 
year-over-year. Net income reached $680 million, up 31% from the previous year. The company's AI 
division was the primary growth driver, contributing $1.8 billion (43%) of total revenue.

PRODUCT DIVISIONS
The AI Solutions division launched three new enterprise products: AutoAnalyze (automated data 
analytics), SmartForecast (demand prediction), and DocIntel (document intelligence). AutoAnalyze 
alone generated $520 million in its first year, exceeding projections by 40%.

The Cloud Infrastructure division generated $1.4 billion in revenue with a 92% client retention rate. 
The division expanded to 15 global data center regions, including new facilities in Singapore and 
Frankfurt.

The Enterprise Software division contributed $1.0 billion, with the flagship ERP product serving 
over 2,300 enterprise clients globally.

WORKFORCE
TechCorp employed 12,400 people at year end, with 3,200 in R&D (26% of workforce). The company 
hired 2,100 new employees in 2024, with 45% in AI-related roles. Employee satisfaction score 
was 4.2/5.0, and voluntary turnover decreased to 8.3% from 11.1% in 2023.

FINANCIAL HIGHLIGHTS
- Revenue: $4.2B (+23% YoY)
- Net Income: $680M (+31% YoY)
- Operating Margin: 22.4%
- Free Cash Flow: $890M
- R&D Spending: $620M (14.8% of revenue)
- Total Assets: $8.1B
- Total Debt: $1.2B (debt-to-equity ratio: 0.28)

STRATEGIC OUTLOOK 2025
TechCorp plans to invest $800M in AI R&D in 2025, targeting 30% revenue growth. Key initiatives 
include launching TechCorp AI Studio (a no-code AI platform), expanding into healthcare AI, and 
opening a new R&D center in Tokyo. The company projects revenue of $5.4-5.6 billion for FY2025.

SUSTAINABILITY
TechCorp achieved carbon neutrality in operations in Q3 2024. The company runs 78% of its data 
centers on renewable energy, with a target of 100% by 2026. Water usage in data centers was 
reduced by 34% through advanced cooling technology.

BOARD OF DIRECTORS
CEO: Sarah Chen (appointed 2021)
CFO: Marcus Williams
CTO: Dr. Priya Patel
Board Chair: Robert Kim
Board members: Jennifer Lee, David Okafor, Maria Santos, Thomas Wright
"""

print(f"📄 Document loaded: {len(company_report)} characters")
print(f"   (~{len(company_report.split())} words)")

In [ ]:
# ============================================================
# ✂️ Split into Chunks
# ============================================================
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # Max characters per chunk
    chunk_overlap=50,      # Overlap between chunks (preserves context)
    separators=["\n\n", "\n", ". ", " "],  # Split on these boundaries (in order)
)

chunks = splitter.split_text(company_report)

print(f"✂️ Split into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks[:3]):
    print(f"📦 Chunk {i+1} ({len(chunk)} chars):")
    print(f"   {chunk[:100]}...")
    print()

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> Chunk size is a critical parameter. Too small = chunks lack context. Too large = irrelevant information dilutes the good stuff. <b>300-500 characters</b> with <b>50-100 overlap</b> is a good starting point for most documents.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Embed & Store in ChromaDB</h2>
</div>

In [ ]:
# ============================================================
# 🧮 Embed Chunks & Store in ChromaDB
# ============================================================
import chromadb

chroma_client = chromadb.Client()

# Delete if exists, then create fresh
try:
    chroma_client.delete_collection("techcorp_report")
except:
    pass

collection = chroma_client.create_collection(name="techcorp_report")

# Embed all chunks
emb_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks
)
chunk_embeddings = [item.embedding for item in emb_response.data]

# Add to ChromaDB
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"chunk_index": i, "char_count": len(c)} for i, c in enumerate(chunks)]
)

print(f"✅ Stored {collection.count()} chunks in ChromaDB")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Build the RAG Chain</h2>
</div>

Now we connect it all: **retrieve relevant chunks** → **inject as context** → **generate grounded answer**.

In [ ]:
# ============================================================
# 🔗 RAG Chain: Retrieve → Augment → Generate
# ============================================================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def retrieve(query, n_results=3):
    """Retrieve relevant chunks from ChromaDB."""
    q_emb = client.embeddings.create(model="text-embedding-3-small", input=[query])
    results = collection.query(
        query_embeddings=[q_emb.data[0].embedding],
        n_results=n_results
    )
    return results["documents"][0]

# RAG prompt template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based ONLY on the provided context.
If the context doesn't contain enough information to answer, say "I don't have enough information in the provided documents to answer that."
Always cite which part of the context supports your answer."""),
    ("user", """Context:
---
{context}
---

Question: {question}""")
])

rag_chain = rag_prompt | llm | StrOutputParser()

def ask_with_rag(question):
    """Ask a question using RAG."""
    chunks_found = retrieve(question)
    context = "\n\n".join(chunks_found)
    answer = rag_chain.invoke({"context": context, "question": question})
    return answer, chunks_found

# Test it!
question = "What was TechCorp's revenue in 2024 and what drove the growth?"
answer, sources = ask_with_rag(question)

print(f"❓ Question: {question}\n")
print(f"🤖 RAG Answer:\n   {answer}\n")
print(f"📄 Sources used ({len(sources)} chunks):")
for i, s in enumerate(sources):
    print(f"   {i+1}. {s[:80]}...")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ WITH RAG vs WITHOUT RAG</h2>
</div>

This is the key comparison. Let's ask the same questions with and without RAG to see the difference.

In [ ]:
# ============================================================
# ⚔️ Side-by-Side: WITH RAG vs WITHOUT RAG
# ============================================================

no_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question concisely. If you don't know, say so."),
    ("user", "{question}")
])
no_rag_chain = no_rag_prompt | llm | StrOutputParser()

questions = [
    "Who is the CEO of TechCorp?",
    "How many employees does TechCorp have in R&D?",
    "What is TechCorp's debt-to-equity ratio?",
    "What percentage of data centers run on renewable energy?",
]

print("="*80)
print("⚔️  WITH RAG vs WITHOUT RAG — Side-by-Side Comparison")
print("="*80)

for q in questions:
    # Without RAG
    no_rag_answer = no_rag_chain.invoke({"question": q})

    # With RAG
    rag_answer, _ = ask_with_rag(q)

    print(f"\n❓ {q}")
    print(f"   ❌ Without RAG: {no_rag_answer[:120]}")
    print(f"   ✅ With RAG:    {rag_answer[:120]}")
    print(f"   {'─'*70}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> Without RAG, the LLM either <b>hallucinates</b> (makes up answers) or says "I don't know" — because TechCorp is a fictional company not in its training data. With RAG, every answer is <b>grounded in the actual document</b>. This is why RAG is essential for enterprise AI applications.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">6️⃣ Evaluate Retrieval Quality</h2>
</div>

Not all retrieved chunks are equally useful. Let's look at what gets retrieved and evaluate quality.

In [ ]:
# ============================================================
# 📊 Retrieval Quality Analysis
# ============================================================

test_queries = [
    ("What is TechCorp's revenue?", "Financial / Executive Summary"),
    ("Tell me about sustainability efforts", "Sustainability section"),
    ("What products does the AI division sell?", "Product Divisions"),
    ("What are TechCorp's plans for 2025?", "Strategic Outlook"),
]

print("📊 Retrieval Quality Check\n")

for query, expected in test_queries:
    q_emb = client.embeddings.create(model="text-embedding-3-small", input=[query])
    results = collection.query(
        query_embeddings=[q_emb.data[0].embedding],
        n_results=3,
        include=["documents", "distances"]
    )

    print(f"🔍 Query: '{query}'")
    print(f"   Expected section: {expected}")
    for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
        sim = 1 - dist
        bar = "█" * int(sim * 30)
        print(f"   {i+1}. [{sim:.2f}] {bar}")
        print(f"      {doc[:80]}...")
    print()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "Why do we split documents into chunks before embedding?",
        "options": [
            "To reduce API costs",
            "Because embedding models have a token limit, and smaller chunks allow more precise retrieval",
            "To make the document easier to read",
            "Because ChromaDB only accepts short strings"
        ],
        "answer": 1,
        "explanation": "Chunking serves two purposes: (1) embedding models have token limits, and (2) smaller chunks mean the retrieved context is more focused and relevant, reducing noise in the LLM's input."
    },
    {
        "q": "What is the main advantage of RAG over fine-tuning for enterprise knowledge?",
        "options": [
            "RAG is always more accurate",
            "RAG doesn't require any LLM calls",
            "RAG works with documents that change frequently — no retraining needed",
            "RAG is cheaper to run"
        ],
        "answer": 2,
        "explanation": "RAG's key advantage is that when your documents change (new reports, updated policies), you just update the vector store — no expensive retraining. Fine-tuning bakes knowledge into the model weights, which is static."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Build RAG for Your Own Document

Paste your own text (or upload a file) and build a RAG pipeline for it. Replace each `-----` with the correct value.

In [ ]:
# ============================================================
# 🔧 Exercise: Build RAG for Your Own Document
# ============================================================
# Replace each ----- with the correct value

# Step 1: Your document (paste text or load from URL)
my_document = """-----"""  # YOUR DOCUMENT TEXT HERE (at least 500 characters)

# Step 2: Split into chunks
my_splitter = RecursiveCharacterTextSplitter(
    chunk_size=-----,        # Pick a chunk size (200-500)
    chunk_overlap=-----,     # Pick an overlap (30-100)
)
my_chunks = my_splitter.split_text(my_document)
print(f"✂️ Split into {len(my_chunks)} chunks")

# Step 3: Embed and store
try:
    chroma_client.delete_collection("my_rag")
except:
    pass
my_coll = chroma_client.create_collection(name="my_rag")

my_embs = client.embeddings.create(model="-----", input=my_chunks)  # Which model?
my_vectors = [item.embedding for item in my_embs.data]

my_coll.add(
    documents=my_chunks,
    embeddings=my_vectors,
    ids=[f"c_{i}" for i in range(len(my_chunks))]
)

# Step 4: Query!
my_question = "-----"  # YOUR QUESTION about your document

q_emb = client.embeddings.create(model="text-embedding-3-small", input=[my_question])
results = my_coll.query(query_embeddings=[q_emb.data[0].embedding], n_results=3)
context = "\n\n".join(results["documents"][0])

answer = rag_chain.invoke({"context": context, "question": my_question})
print(f"\n❓ {my_question}")
print(f"🤖 {answer}")

**📝 Your Observations:** *(double-click to edit this cell)*

1. Was the RAG answer accurate based on your document? _[Your answer]_

2. Did the retrieved chunks contain the right information? _[Your answer]_

3. What happens if you ask a question that's NOT in your document? Does the system correctly say it doesn't know? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions:</p>
  <ol style="font-size: 14px;">
    <li>Change chunk_size to 100 vs 500 — how does answer quality change?</li>
    <li>Upload a <b>PDF</b> using <code>PyPDF2</code> and build RAG on it</li>
    <li>Try a question that requires information from <b>multiple chunks</b> — does it combine them correctly?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've built a complete RAG pipeline and seen how it eliminates hallucination with document grounding.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li>RAG pipeline: <b>Load → Split → Embed → Store → Retrieve → Generate</b></li>
      <li><code style="color: #90caf9;">RecursiveCharacterTextSplitter</code> chunks documents with smart boundaries</li>
      <li>RAG answers are <b>grounded in real documents</b> — no hallucination</li>
      <li>Chunk size and overlap are critical parameters to tune</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M06 Lab 1: Agents &amp; LangGraph</p>
</div>